# Creating Cherry Rainbow Tables

For N = 2**16, currently only making one table

## Imports

In [1]:
import pickle
import random
from hashlib import sha256
import mmh3
from tqdm import tqdm
from math import pi, sqrt, e, log

## Table Parameters

### Initialise Startpoints

In [2]:
# initialise startpoints - either generate them or load from pickle - to keep same across runs
def get_startpoints(N, m_0, nlabel, alpha):
    # try opening pickle file, else generate and save
    try:
        with open(f'startpoints_N_{nlabel}_alpha_{alpha}.pkl', 'rb') as f:
            startpoints = pickle.load(f)

    # no file found - generate and save
    except FileNotFoundError:
        # random but unique - store as a set?
        startpoints = set()
        while len(startpoints) < m_0:
            startpoints.add(random.randint(0, N-1))

        # store startpoints in pickle file
        with open(f'startpoints_N_{nlabel}_alpha_{alpha}.pkl', 'wb') as f:
            pickle.dump(startpoints, f)

    # return the startpoints
    return startpoints

### Parameters

In [3]:
# label N to find easier - label is the exponent
nlabel = 16
##############################################################################
N = 2 ** 16 # keyspace
p = 1 - e ** -2 # our table coverage - 86%
##############################################################################
t = round(log(1-p)/log(1-N**(-1/3))) # chain length t
alpha = 0.95 # maximality factor
mt_target = N**(2/3) # our target mt
m_0 = round(mt_target/(1-alpha))    # m_0 - number of startpoints
##############################################################################
# initialise startpoints 
startpoints = get_startpoints(N, m_0, nlabel, alpha)
##############################################################################
# cost factor used for Kis
cost = 5

### Cherry-picks per column - Kis

In [4]:
# get # cherry-picks per column from pickle file 

# load pickle file
with open(f'higher_costs_ftol_1.pickle', 'rb') as f:
    data = pickle.load(f)

# structure of file
# array of different alpha used
    # for each alpha, array of different costs used
    # for each cost, array arranged as [Kjs, m_0, final_cost, m_values]

# to get Kjs for alpha = 0.95 and cost factor = 5
Kis = data[-1][0][0]

## Hash and Reduction Functions

In [5]:
# # Hash function
# def H(x):
# 	return int.from_bytes(sha256(x.to_bytes(8)).digest())

# # Reduction function
# # currently mod but should change to murmurhash in future
# # def r(y, i, ell=0):   # also takes in ell - number of tables - for future use (but currently ell=0)
# # 	return (y + i + ell*t) % N

# def r(y, i, ell=0):
# 	# use index as seed and do mod N at the end
# 	return mmh3.hash(y.to_bytes(32, 'little'), i + ell*t, signed=False) % N

def H(x):
    return sha256(x.to_bytes(8, 'little')).digest()  # return bytes directly

def r(y, i, ell=0):
    return mmh3.hash(y, i + ell*t, signed=False) % N


## Building the Table

In [6]:
# build cherry table
def build_cherry_table(t, alpha, startpoints, Kis, cost):
    # store table in dictionary
    # initialise the table with sp:sp pairs
    table = {sp: sp for sp in startpoints}  # store all the points then remove duplicate entries - can't do duplicate keys in dictionary anyway so we can just store all ep:sp

    # Instead of making the table chain by chain, we have to make it column by column to test what reduction function to choose
    # store reduction function indexes
    rf_indexes = []

    # for each column
    for i in tqdm(range(t), desc=f"Calculating columns: "):

        # variable to store best cherry-pick
        best_trial = -1

        # hash all current points then store with startpoints - this stores all our current points
        hashed_points = {H(mi): sp for mi, sp in table.items()}

        # we are going to continuously replace table with the best rf trial, so we empty it for now
            # we haven't lost the current points as we have them hashed in the hashed_points dictionary
        table = {}

        # get # cherry-picks for this column
        k_i = round(Kis[i])

        # trial all the reduction functions for this column
        # for rf_trial in tqdm(range(k_i), desc=f"Choosing best RF: "):
        for rf_trial in range(k_i):

            # create a trial column to store results of current trial
            trial_column = {}

            # go through each key in hashed_points and store its reduction with sp
            for x in hashed_points:
                # reduce the hash and store in column
                trial_column[r(x, rf_trial)] = hashed_points[x]

            # if trial_column is bigger than current table stored, we replace it
            if len(trial_column) > len(table):
                # replace it 
                table = trial_column
                # replace best cherry-pick
                best_trial = rf_trial

        # now store the best rf cherry pick
        rf_indexes.append(best_trial)

    # # store the table as a pickle file
    # with open(f'cherry_table_alpha_{alpha}_t_{t}_cost_{cost}.pkl', 'wb') as f:
    #     pickle.dump(table, f)
    #     f.close()

    # # store the indexes as a pickle file
    # with open(f'cherry_indexes_alpha_{alpha}_t_{t}_cost_{cost}.pkl', 'wb') as f:
    #     pickle.dump(rf_indexes, f)
    #     f.close()


    # finished, so return table and rf indexes
    return table, rf_indexes

## Searching the Table

In [23]:
# function to continue search
# we need to take in what column we're at (c), our key (y), # columns (t), current total of hashes and reductions
def continue_cherry_search(y, t, c, hashes, reductions, indexes):
    # reduce c by 1 to move to the previous column
    c -= 1
    # number of columns between current column and end 
    # diff = t - c  
    # reduce y by the new c index
    x = r(y, indexes[c])
    reductions += 1
    # then hash and reduce however many times to move back through columns
    for i in range((t - c) - 1, 0, -1):    # decrease difference by 1 (as we already reduced by current diff index) then continuously reduce by 1
        x = r(H(x), indexes[t-i])
        hashes += 1
        reductions += 1

    # return x, c, hashes, reductions
    return x, c, hashes, reductions

In [ ]:
def search_cherry_table(y, t, table, indexes):
    # keep track of hashes and reductions
    col_round = 0
    hashes = 0
    reductions = 0

    # keep track of column we're in 
    c = t - 1
    # reduce (r_t-1) the hash then compare in the table 
    x = r(y, indexes[c])
    reductions += 1

    # while we haven't reached the end of our chain
    while c > -1:
        # if there is a match in the keys (our endpoints), regenerate chain until we find the key
        if (x in table.keys()):
            point = table[x]   # search for endpoint in table and get startpoint
            # regenerate chain until column c - we are now in the column before the match
            for i in range(c):
                point = r(H(point), indexes[i])
                hashes += 1
                reductions += 1
            
            # hash the point - if it is a match we have found our preimage 
            if H(point) == y:
                hashes += 1
                return point, hashes, reductions, col_round
            
            # if that didn't work then we ran into a false alarm
            else:
                # print("false alarm")
                # continue the search
                x, c, hashes, reductions = continue_cherry_search(y, t, c, hashes, reductions, indexes)
 
        # if we didn't find a match in endpoints, we need to restart the search
        else:
            # hash and reduce the ciphertext accordingly
            x, c, hashes, reductions = continue_cherry_search(y, t, c, hashes, reductions, indexes)

        col_round += 1

    # We have searched all columns - return -1
    return -1, hashes, reductions, col_round

## Run

In [7]:
def run_cherry_simulation(N, t, alpha):
    # first generate starpoints and store them
    mt_max = (2*N)/(t+2)
    mt_target = alpha * mt_max
    cherry_m_0 = round(mt_target/(1-alpha))    # our starting m_0

    # generate sample sets of starpoints
    starpoints = set()
    while len(starpoints) < (cherry_m_0):
        starpoints.add(random.randint(0, N-1))

    # get the kis for this alpha
    with open("simulations/cherry_N_16_alpha_0.5_kis.pkl", "rb") as f:
        optimisation_kis = pickle.load(f)

    # build cherry table
    table, rf_indexes = build_cherry_table(t, alpha, starpoints, optimisation_kis, cost)

    return table, rf_indexes

table_test, rf_test = run_cherry_simulation(N, t, 0.5)
print(f"Table size: {len(table_test)}")

Calculating columns: 100%|██████████| 80/80 [00:59<00:00,  1.34it/s]

Table size: 1268


### Precomputation Phase - Build the table

In [25]:
# either build or load table
def get_cherry_table():
    # try loading table from pickle file
    try:
        with open(f'cherry_table_alpha_{alpha}_t_{t}_cost_{cost}.pkl', 'rb') as f:
            table = pickle.load(f)
            f.close()

        with open(f'cherry_indexes_alpha_{alpha}_t_{t}_cost_{cost}.pkl', 'rb') as f:
            indexes = pickle.load(f)
            f.close()

    # if no pickle file found, build the table
    except FileNotFoundError:
        table, indexes = build_cherry_table(t, alpha, startpoints, Kis, cost)

    return table, indexes


In [26]:
cherry, indexes = get_cherry_table()

In [29]:
# check if all items in a chain can be found
# regenerate chain
chain = []
# get the first startpoint that was stored
p = next(iter(cherry.values()))
chain.append(p)

for i in range(t):
    # hash then reduce
    p = r(H(p), indexes[i])
    # store point in chain
    chain.append(p)

# now test all the items in the chain and see if they can be searched
for i in range(1, len(chain)):  # i = 1 to 79
    print(f"Column {t - i}:")       # 
    print(f"     key = {chain[-(1+i)]}")
    y = H(chain[-(1+i)])
    value, hashes, reductions, col_round = search_cherry_table(y, t, cherry, indexes)
    print(f"     {value}")
    print(f'     Hashes: {hashes}, Reductions: {reductions}')

Column 79:
     key = 50675
57640
     50675
     Hashes: 80, Reductions: 80
Column 78:
     key = 24292
57640
     24292
     Hashes: 80, Reductions: 81
Column 77:
     key = 62002
57640
     62002
     Hashes: 159, Reductions: 161
Column 76:
     key = 55878
57640
     55878
     Hashes: 83, Reductions: 86
Column 75:
     key = 47513
57640
     47513
     Hashes: 86, Reductions: 90
Column 74:
     key = 53709
57640
     53709
     Hashes: 90, Reductions: 95
Column 73:
     key = 51020
57640
     51020
     Hashes: 172, Reductions: 178
Column 72:
     key = 5948
56734
     5948
     Hashes: 83, Reductions: 86
Column 71:
     key = 23961
57640
     23961
     Hashes: 181, Reductions: 189
Column 70:
     key = 2993
57640
     2993
     Hashes: 188, Reductions: 197
Column 69:
     key = 40257
57640
     40257
     Hashes: 196, Reductions: 206
Column 68:
     key = 24397
57640
     24397
     Hashes: 358, Reductions: 369
Column 67:
     key = 8002
26843
     8002
     Hashes: 81, Reductio

In [31]:
# point = chain[-1]
point = 57640
print(f"searching for {point}")

y = H(point)

value, hashes, reductions, col_round = search_cherry_table(y, t, cherry, indexes)
print(value, hashes, reductions, col_round)



searching for 57640
57640
57640 5259 5338 79
